In [1]:
import pandas as pd
import numpy as np

# Load clean CSV for all analysis except race imputation
df = pd.read_csv("brfss_clean_2020_2024.csv")

# Established mappings — consistent with preprocess notebook
race_map = {
    1: "NH-White", 2: "NH-Black", 3: "AIAN",
    4: "Asian", 5: "NHOPI", 6: "Other/Multiracial",
    7: "Hispanic"
}

age_map = {
    1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39",
    5: "40-44", 6: "45-49", 7: "50-54", 8: "55-59",
    9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"
}

sex_map = {1: "Male", 2: "Female"}

education_map = {
    1: "Did not graduate high school",
    2: "Graduated high school",
    3: "Attended college or technical school",
    4: "Graduated college or technical school"
}

income_map = {
    1: "<15k", 2: "15k-25k", 3: "25k-35k",
    4: "35k-50k", 5: "50k-100k", 6: "100k-200k", 7: "200k+"
}

df["race_group"]   = df["_RACEPRV"].map(race_map)
df["age_group"]    = df["_AGEG5YR"].map(age_map)
df["sex"]          = df["_SEX"].map(sex_map)
df["education"]    = df["_EDUCAG"].map(education_map)
df["income_group"] = df["_INCOMG1"].map(income_map)

print("Shape:", df.shape)
print("Years:", sorted(df["year"].unique()))
print("Race groups:", sorted(df["race_group"].dropna().unique()))
print("Income groups:", sorted(df["income_group"].dropna().unique()))

Shape: (1622499, 16)
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Race groups: ['AIAN', 'Asian', 'Hispanic', 'NH-Black', 'NH-White', 'NHOPI', 'Other/Multiracial']
Income groups: ['100k-200k', '15k-25k', '200k+', '25k-35k', '35k-50k', '50k-100k', '<15k']


In [2]:
print("=" * 60)
print("INCOME: COST/BENEFIT ANALYSIS")
print("=" * 60)

# Row counts by year
print("\nRow counts by year:")
print(df["year"].value_counts().sort_index())

# How many rows have valid income per year
print("\nValid income rows by year (excluding missing code 9):")
valid_income = df[df["_INCOMG1"] < 8]
print(valid_income["year"].value_counts().sort_index())

print("\n--- Option A: Drop 2020 income ---")
rows_2020 = len(df[df["year"] == 2020])
rows_2021_2024 = len(df[df["year"] != 2020])
print(f"  2020 rows kept but income = NaN: {rows_2020:,}")
print(f"  Income analysis uses 2021-2024: {rows_2021_2024:,} rows")

print("\n--- Option B: Collapse to 5 bins (50k+) ---")
print(f"  All 5 years usable: {len(df):,} rows")
print(f"  BUT codes 5+6+7 get merged into one bin")

# Who is in high income brackets (codes 5,6,7) in 2021-2024
df_2124 = df[(df["year"] != 2020) & (df["_INCOMG1"] < 8)].copy()
df_2124["high_income"] = df_2124["_INCOMG1"].isin([5, 6, 7]).astype(int)

def weighted_pct(group):
    return np.average(group["high_income"], weights=group["_LLCPWT"])

print("\n--- Who loses most if we collapse to 5 bins (weighted % earning 50k+) ---")
print("\nBy Education:")
print(df_2124.groupby("education").apply(weighted_pct)
      .sort_values(ascending=False).round(3))

print("\nBy Race:")
print(df_2124.groupby("race_group").apply(weighted_pct)
      .sort_values(ascending=False).round(3))

print("\nBy Age:")
print(df_2124.groupby("age_group").apply(weighted_pct)
      .sort_index().round(3))

INCOME: COST/BENEFIT ANALYSIS

Row counts by year:
year
2020    300259
2021    320968
2022    326682
2023    326419
2024    348171
Name: count, dtype: int64

Valid income rows by year (excluding missing code 9):
year
2020    300259
2021    320968
2022    326682
2023    326419
2024    348171
Name: count, dtype: int64

--- Option A: Drop 2020 income ---
  2020 rows kept but income = NaN: 300,259
  Income analysis uses 2021-2024: 1,322,240 rows

--- Option B: Collapse to 5 bins (50k+) ---
  All 5 years usable: 1,622,499 rows
  BUT codes 5+6+7 get merged into one bin

--- Who loses most if we collapse to 5 bins (weighted % earning 50k+) ---

By Education:
education
Graduated college or technical school    0.826
Attended college or technical school     0.590
Graduated high school                    0.453
Did not graduate high school             0.204
dtype: float64

By Race:
race_group
Asian                0.705
NH-White             0.663
Other/Multiracial    0.571
NHOPI                0.55

In [3]:
print("=" * 60)
print("WEIGHTED OBESITY RATES BY DEMOGRAPHIC GROUP")
print("=" * 60)

def weighted_obesity(group):
    return np.average(group["obese"], weights=group["_LLCPWT"])

print("\nBy Year:")
print(df.groupby("year").apply(weighted_obesity).round(4))

print("\nBy Sex:")
print(df.groupby("sex").apply(weighted_obesity)
      .sort_values(ascending=False).round(4))

print("\nBy Race:")
print(df.groupby("race_group").apply(weighted_obesity)
      .sort_values(ascending=False).round(4))

print("\nBy Education:")
print(df.groupby("education").apply(weighted_obesity)
      .sort_values(ascending=False).round(4))

print("\nBy Income (2021-2024 only, 2020 not comparable):")
print(df[df["year"] != 2020].groupby("income_group").apply(weighted_obesity)
      .sort_index().round(4))

print("\nBy Age:")
print(df.groupby("age_group").apply(weighted_obesity)
      .sort_index().round(4))

WEIGHTED OBESITY RATES BY DEMOGRAPHIC GROUP

By Year:
year
2020    0.3285
2021    0.3404
2022    0.3446
2023    0.3384
2024    0.3415
dtype: float64

By Sex:
sex
Female    0.3486
Male      0.3292
dtype: float64

By Race:
race_group
NH-Black             0.4349
AIAN                 0.4010
NHOPI                0.3746
Hispanic             0.3681
Other/Multiracial    0.3348
NH-White             0.3299
Asian                0.1319
dtype: float64

By Education:
education
Did not graduate high school             0.3833
Attended college or technical school     0.3672
Graduated high school                    0.3636
Graduated college or technical school    0.2767
dtype: float64

By Income (2021-2024 only, 2020 not comparable):
income_group
100k-200k    0.3238
15k-25k      0.3744
200k+        0.2322
25k-35k      0.3676
35k-50k      0.3575
50k-100k     0.3485
<15k         0.3794
dtype: float64

By Age:
age_group
18-24    0.2014
25-29    0.2973
30-34    0.3426
35-39    0.3599
40-44    0.3840
45-49   

In [ ]:
print("=" * 60)
print("RACE: IMPUTATION ANALYSIS")
print("=" * 60)
print("How _RACEPRV fills missing values in _RACE")
print("Requires raw XPT files — this info is not in the clean CSV")
print()

files = { 
    2020: "LLCP2020.XPT",
    2021: "LLCP2021.XPT",
    2022: "LLCP2022.XPT",
    2023: "LLCP2023.XPT",
    2024: "LLCP2024.XPT"
}

# Original 8-category labels for raw _RACEPRV before recoding
raw_race_labels = {
    1: "NH-White", 2: "NH-Black", 3: "AIAN",
    4: "Asian", 5: "NHOPI", 6: "Other",
    7: "Multiracial", 8: "Hispanic"
}

for year, f in files.items():
    df_temp = pd.read_sas(f, format="xport", encoding="latin1")

    if year == 2022:
        raw_col, calc_col = "_RACE1", "_RACEPR1"
    else:
        raw_col, calc_col = "_RACE", "_RACEPRV"

    missing = df_temp[df_temp[raw_col] == 9]
    total_missing = len(missing)
    total_rows = len(df_temp)

    print(f"{year}:")
    print(f"  Total rows: {total_rows:,}")
    print(f"  {raw_col} missing (code 9): {total_missing:,} ({total_missing/total_rows*100:.1f}%)")
    print(f"  How {calc_col} filled those {total_missing:,} rows:")
    imputed = missing[calc_col].value_counts().sort_index()
    for code, count in imputed.items():
        label = raw_race_labels.get(int(code), "Unknown")
        print(f"    {label}: {count:,} ({count/total_missing*100:.1f}%)")
    print()

RACE: IMPUTATION ANALYSIS
How _RACEPRV fills missing values in _RACE
Requires raw XPT files — this info is not in the clean CSV

2020:
  Total rows: 401,958
  _RACE missing (code 9): 8,987 (2.2%)
  How _RACEPRV filled those 8,987 rows:
    NH-White: 7,986 (88.9%)
    NH-Black: 259 (2.9%)
    AIAN: 141 (1.6%)
    Asian: 105 (1.2%)
    Other: 410 (4.6%)
    Hispanic: 86 (1.0%)

2021:
  Total rows: 438,693
  _RACE missing (code 9): 10,990 (2.5%)
  How _RACEPRV filled those 10,990 rows:
    NH-White: 9,517 (86.6%)
    NH-Black: 360 (3.3%)
    AIAN: 145 (1.3%)
    Asian: 149 (1.4%)
    Other: 527 (4.8%)
    Hispanic: 292 (2.7%)

2022:
  Total rows: 445,132
  _RACE1 missing (code 9): 14,055 (3.2%)
  How _RACEPR1 filled those 14,055 rows:
    NH-White: 13,092 (93.1%)
    NH-Black: 430 (3.1%)
    AIAN: 168 (1.2%)
    Asian: 140 (1.0%)
    Other: 165 (1.2%)
    Multiracial: 60 (0.4%)

2023:
  Total rows: 433,323
  _RACE missing (code 9): 9,484 (2.2%)
  How _RACEPRV filled those 9,484 rows:
    

In [5]:
print("=" * 60)
print("SUMMARY FOR TEAM DECISION")
print("=" * 60)

print("""
INCOME
------
2020 uses _INCOMG (5 bins, max 50k+)
2021-2024 use _INCOMG1 (7 bins, up to 200k+)
These are NOT comparable.

Option A — Drop 2020 income:
  Keep all 1,622,499 rows but 2020 income = NaN
  Income analysis uses 2021-2024 only (1,322,240 rows)
  Full 7-bin detail preserved

Option B — Collapse all years to 5 bins (50k+):
  All 1,622,499 rows usable for income analysis
  BUT codes 5+6+7 merged into one bin
  Groups most affected by losing that granularity:
    College grads:    82.6% earn 50k+ (most detail lost)
    Asian:            70.5% earn 50k+
    NH-White:         66.3% earn 50k+
    Ages 40-54:       67-69% earn 50k+

RACE IMPUTATION
---------------
_RACE missing (code 9): 2-3.2% per year (8,987-14,055 rows)
_RACEPRV fills those using _IMPRACE

Of imputed rows per year:
  83-93% assigned NH-White
  3-6%   assigned NH-Black
  1-5%   split across AIAN, Asian, Other, Hispanic

Implication:
  _RACEPRV slightly inflates NH-White counts
  Effect: max +0.9% of total rows across any year
  Alternative (raw _RACE): leaves 2-3.2% of rows
  with no race label at all

WEIGHTED OBESITY RATES (for reference)
---------------------------------------
Overall range: 32.85% (2020) to 34.46% (2022)
Highest by race:  NH-Black (43.5%), AIAN (40.1%)
Lowest by race:   Asian (13.2%)
Highest by age:   50-54 (40.7%)
Lowest income:    <15k (37.9%), highest: 200k+ (23.2%)

DECISIONS NEEDED:
  1. Income: Option A (drop 2020) or Option B (collapse to 5 bins)?
  2. Race: Accept _RACEPRV with documented NH-White inflation?
""")

SUMMARY FOR TEAM DECISION

INCOME
------
2020 uses _INCOMG (5 bins, max 50k+)
2021-2024 use _INCOMG1 (7 bins, up to 200k+)
These are NOT comparable.

Option A — Drop 2020 income:
  Keep all 1,622,499 rows but 2020 income = NaN
  Income analysis uses 2021-2024 only (1,322,240 rows)
  Full 7-bin detail preserved

Option B — Collapse all years to 5 bins (50k+):
  All 1,622,499 rows usable for income analysis
  BUT codes 5+6+7 merged into one bin
  Groups most affected by losing that granularity:
    College grads:    82.6% earn 50k+ (most detail lost)
    Asian:            70.5% earn 50k+
    NH-White:         66.3% earn 50k+
    Ages 40-54:       67-69% earn 50k+

RACE IMPUTATION
---------------
_RACE missing (code 9): 2-3.2% per year (8,987-14,055 rows)
_RACEPRV fills those using _IMPRACE

Of imputed rows per year:
  83-93% assigned NH-White
  3-6%   assigned NH-Black
  1-5%   split across AIAN, Asian, Other, Hispanic

Implication:
  _RACEPRV slightly inflates NH-White counts
  Effect: 

In [6]:
print("=" * 60)
print("SIMULATION: OPTION A vs OPTION B")
print("=" * 60)

# Option A — drop 2020 income, use 7 bins for 2021-2024
df_A = df[df["year"] != 2020].copy()
df_A = df_A[df_A["_INCOMG1"] < 8]

income_map_7 = {
    1: "<15k", 2: "15k-25k", 3: "25k-35k",
    4: "35k-50k", 5: "50k-100k", 6: "100k-200k", 7: "200k+"
}
df_A["income_bin"] = df_A["_INCOMG1"].map(income_map_7)

obesity_A = df_A.groupby("income_bin").apply(weighted_obesity).sort_index()

# Option B — collapse to 5 bins for all years
df_B = df[df["_INCOMG1"] < 8].copy()
income_map_5 = {
    1: "<15k", 2: "15k-25k", 3: "25k-35k",
    4: "35k-50k", 5: "50k+", 6: "50k+", 7: "50k+"
}
df_B["income_bin"] = df_B["_INCOMG1"].map(income_map_5)

obesity_B = df_B.groupby("income_bin").apply(weighted_obesity).sort_index()

print("\nOption A — 7 bins, 2021-2024 only:")
for bin_name, rate in obesity_A.items():
    print(f"  {bin_name:15s}: {rate:.4f}")
print(f"  Range: {obesity_A.max() - obesity_A.min():.4f}")
print(f"  Std dev: {obesity_A.std():.4f}")

print("\nOption B — 5 bins, all years:")
for bin_name, rate in obesity_B.items():
    print(f"  {bin_name:15s}: {rate:.4f}")
print(f"  Range: {obesity_B.max() - obesity_B.min():.4f}")
print(f"  Std dev: {obesity_B.std():.4f}")

print("\nConclusion:")
print(f"  Option A std dev: {obesity_A.std():.4f}")
print(f"  Option B std dev: {obesity_B.std():.4f}")
if obesity_A.std() > obesity_B.std():
    print("  Option A preserves MORE variation in obesity rates across income")
else:
    print("  Option B preserves MORE variation in obesity rates across income")

SIMULATION: OPTION A vs OPTION B

Option A — 7 bins, 2021-2024 only:
  100k-200k      : 0.3238
  15k-25k        : 0.3744
  200k+          : 0.2322
  25k-35k        : 0.3676
  35k-50k        : 0.3575
  50k-100k       : 0.3485
  <15k           : 0.3794
  Range: 0.1472
  Std dev: 0.0512

Option B — 5 bins, all years:
  15k-25k        : 0.3702
  25k-35k        : 0.3644
  35k-50k        : 0.3546
  50k+           : 0.3195
  <15k           : 0.3783
  Range: 0.0588
  Std dev: 0.0229

Conclusion:
  Option A std dev: 0.0512
  Option B std dev: 0.0229
  Option A preserves MORE variation in obesity rates across income
